# MNIST demo with CREST


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Save the start time
import time
start_time = time.time()

# Allow crest to be imported
%cd -q ..

#### Load MNIST Datasets


In [3]:
from crest import DataServer

# Load datasets from server
ds_train, ds_valid, ds_test, ds_pred = DataServer.load('mnist')

# Get input/output labels
[INP, OUT] = ds_train.labels

# Num classes
n_classes = 10
patch_size = 28

In [4]:
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, Dropout, Flatten, Dense, Lambda
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph
from crest.model.KerasNode import KerasNode
from crest.model.TensorSpec import TensorSpec

layers = [
    KerasNode(Lambda(lambda x: tf.cast(x, tf.float32)), name='layer_0', inputs={INP: TensorSpec(
        [None, patch_size, patch_size])}, outputs={OUT: TensorSpec([None, 28, 28])}),        # Cast dtype
    KerasNode(Lambda(lambda x: tf.expand_dims(x, axis=-1)), name='layer_1', inputs={INP: TensorSpec(
        [None, patch_size, patch_size])}, outputs={OUT: TensorSpec([None, 28, 28, 1])}),  # Add channel dimension
    KerasNode(Conv2D(10, 3, padding='same', activation='relu'), name='layer_2', inputs={INP: TensorSpec(
        [None, 28, 28, 1])}, outputs={OUT: TensorSpec([None, 28, 28, 10])}),  # Convolutional layer
    KerasNode(Flatten(), name='layer_3', inputs={INP: TensorSpec(
        [None, 28, 28, 10])}, outputs={OUT: TensorSpec([None, 28*28*10])}),  # Flatten
    KerasNode(Dropout(0.3), name='layer_4', inputs={INP: TensorSpec(
        [None, 28*28*10])}, outputs={OUT: TensorSpec([None, 28*28*10])}),  # Dropout
    KerasNode(Dense(n_classes, activation='softmax'), name='layer_5', inputs={
              INP: TensorSpec([None, 28*28*10])}, outputs={OUT: TensorSpec([None, 10])})  # Output layer
]

for i in layers[:-1]:
    i.rename_io(outputs_map={OUT: INP})

graph = HierarchalTensorGraph(
    name='MNIST',
    inputs={INP: TensorSpec([None, patch_size, patch_size])},
    outputs={OUT: TensorSpec([None, 10])}
)

graph.add_edge('input', layers[0])

for i, layer in enumerate(layers[1:]):
    graph.add_edge(layers[i], layers[i+1])

graph.add_edge(layers[-1], 'output')

#### Create the HierarchalTensorGraph


#### Build a model using the HTG


In [5]:
from crest import Model

# Define a custom metric if desired
def custom_metric(y_true, y_pred):
    squared_difference = tf.square(y_true - y_pred)
    return tf.reduce_mean(squared_difference, axis=-1)


# Set build options
build_kwargs = {
    'optimizer': 'Adam',
    'loss': 'sparse_categorical_crossentropy',
    'metrics': ['sparse_categorical_accuracy', custom_metric]
}

# Create anad build a model using the HTG
model = Model(graph)
model.build(**build_kwargs)

/Users/draghun1/Documents/clean/crest/crest/model/Model.py:83: UserWarning: Use Model.compile instead of Model.build
  warnings.warn('Use Model.compile instead of Model.build')


#### Train the model


In [6]:
# Create the training and validation batchers
from crest.data.Batcher import Batcher

batch_kwargs = {
    'batch_size': 200,            # 200 samples in each batch
    'features': [[INP], [OUT]],  # [{'image': ...}, {'class': ...}]
    'repeat': True,           # Yield indefinitely
    'workers': 0               # Number of background processes
}

batch_train = Batcher(ds_train, **batch_kwargs)
batch_kwargs['shuffle'] = False    # Turn shuffle off for prediction
batch_valid = Batcher(ds_valid,  **batch_kwargs)

with batch_train as train, batch_valid as valid:
    model.fit(train, **{
        'validation_data': valid,
        'steps_per_epoch': 250,  # 50k samples / 200 samples per batch
        'validation_steps': 50,  # 10k samples / 200 samples per batch
        'epochs': 5,
        'verbose': 1
    })

Epoch 1/5


  1/250 [..............................] - ETA: 1:37 - loss: 48.7294 - sparse_categorical_accuracy: 0.1050 - custom_metric: 27.9438

  5/250 [..............................] - ETA: 4s - loss: 31.9471 - sparse_categorical_accuracy: 0.3060 - custom_metric: 27.1911  

  8/250 [..............................] - ETA: 4s - loss: 25.0066 - sparse_categorical_accuracy: 0.4106 - custom_metric: 27.8741

 11/250 [>.............................] - ETA: 4s - loss: 21.0818 - sparse_categorical_accuracy: 0.4841 - custom_metric: 27.4226

...

239/250 [===========================>..] - ETA: 0s - loss: 0.1267 - sparse_categorical_accuracy: 0.9695 - custom_metric: 27.4248

242/250 [============================>.] - ETA: 0s - loss: 0.1267 - sparse_categorical_accuracy: 0.9696 - custom_metric: 27.3902

245/250 [============================>.] - ETA: 0s - loss: 0.1270 - sparse_categorical_accuracy: 0.9695 - custom_metric: 27.3878

248/250 [============================>.] - ETA: 0s - loss: 0.1259 - sparse_categorical_accuracy: 0.9697 - custom_metric: 27.3802

250/250 [==============================] - 10s 40ms/step - loss: 0.1263 - sparse_categorical_accuracy: 0.9696 - custom_metric: 27.3886 - val_loss: 0.1818 - val_sparse_categorical_accuracy: 0.9750 - val_custom_metric: 27.4869


#### Evaluate the model


In [7]:
# Evaluate the model on the test set
batch_test = Batcher(ds_test, **batch_kwargs)
evaluate_kwargs = {
    'verbose': 1,
    'return_dict': True
}

with batch_test as batcher:
    res = model.evaluate(batcher, verbose=1, return_dict=1)

1/1 [==============================] - ETA: 0s - loss: 0.0463 - sparse_categorical_accuracy: 0.9850 - custom_metric: 26.3272

1/1 [==============================] - 0s 26ms/step - loss: 0.0463 - sparse_categorical_accuracy: 0.9850 - custom_metric: 26.3272


Accuracy on the full test set should be > ~97%.


### Make predictions


In [8]:
# Create batcher for predictions
batch_pred = Batcher(ds_pred, **{  # Reset batch iterator
    'batch_size': 200,
    'features': [INP],
    'repeat': True,
    'workers': 0,
    'shuffle': False
})

# Set kwargs for Model.predict() (can be the same as Keras.predict)
pred_kwargs = {
    'batch_size': 200,
    'verbose': False,
    'steps': 10000//200
}

# Generate predictions
with batch_pred as batcher:
    model_pred_batch = model.predict(
        batch_pred, **pred_kwargs)  # Using a batcher
    model_pred_dataset = model.predict(
        ds_pred, **pred_kwargs)   # Using a dataset
    print('Difference:',
          (model_pred_batch['class'] - model_pred_dataset['class']).sum())

Difference: 0.0


#### Adding additional output (e.g., lat and lon) when making a predictions


In [9]:
# Create inputs with additional features lat and lon
import numpy as np
from crest.data.loading import StructuredDataset
x = ds_pred.data[0]
lat = np.arange(len(x), dtype=float)
lon = np.arange(len(x), dtype=float)
x_aux = (lat, lon, x)
ds_aux = StructuredDataset(*x_aux, labels=['lat', 'lon', INP])

# Specify additional output as coords when making predictions
model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)

{'class': array([[1.29823334e-17, 3.82969889e-20, 6.72675958e-15, ...,
         1.00000000e+00, 7.66057074e-16, 3.54507847e-11],
        [2.63829462e-11, 6.88320720e-11, 9.99992132e-01, ...,
         1.00838225e-28, 9.77150693e-14, 1.19844026e-18],
        [1.63723737e-08, 9.99928474e-01, 1.04292849e-05, ...,
         8.39476328e-08, 5.93972181e-05, 7.25199900e-10],
        ...,
        [1.56446882e-18, 2.08343751e-17, 1.84620601e-22, ...,
         1.24430022e-10, 3.92080500e-11, 6.80289047e-10],
        [8.68084382e-10, 1.61795355e-14, 1.83265621e-12, ...,
         2.08326156e-08, 5.07611185e-02, 1.31022715e-09],
        [8.89660185e-14, 2.11368138e-17, 8.21856960e-14, ...,
         3.73656632e-18, 5.06845705e-13, 2.46390535e-17]], dtype=float32),
 'lat': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03]),
 'lon': array([0.000e+00, 1.000e+00, 2.000e+00, ..., 9.997e+03, 9.998e+03,
        9.999e+03])}

In [10]:
print('Elapsed time = ', time.time() - start_time, '[s]')

Elapsed time =  91.86928820610046 [s]


model save and load

In [11]:
model.save()

In [12]:
model.metric.all

['unbiased_rmse',
 'fluctuation_complexity',
 'metric_entropy',
 'relative_error',
 'triple_collocation_error',
 'nse',
 'nse_log',
 'mse',
 'kge',
 'lkg',
 'pearson',
 'alpha_nse',
 'beta_nse',
 'custom_metric']

In [13]:
loaded_model = model.load('crest_cache')

In [14]:
# check that loaded model performs the same as original
result_loaded = loaded_model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)
result = model.predict(ds_aux, coords=['lat', 'lon'], **pred_kwargs)

print('Difference:', (result['class'] - result_loaded['class']).sum())

Difference: 0.0
